## Summary of results in this file

Basically, we are getting order p+1 convergence in the L2 norm for shape functions with order p = 1 for both types of square refinement

initially had big issues with p=2 and p=3 at corners, more notes below but essentially only extrapolate very corner dofs
-	linear extrap worked best for p=2 corners (quad and higher did worse)
-	linear,quadratic, and cubic extrap resulted in only second order convergence for p=3, still need to debug this case quite a bit

In [ ]:
%load_ext autoreload
%autoreload 2

##### load the solver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys  
from tqdm import tqdm
mypath = '/home/bbb/Galerkin-Differencing/general_solve'
sys.path.insert(1, mypath)
from general_solve.variable import SingleComponentVariable as Var
from general_solve.run_convergence import run_it, plot_it

In [ ]:
from general_solve.variable import corner_pin

In [ ]:
doflocs = ['node','cell','xside','yside']
rtypes = ['uniform','stripe','square']
rnames = {'stripe':['vertfinecenter',
					'vertcoarsecenter',
					'horzfinecenter',
					'horzcoarsecenter'],
		  'square':['finecenter','coarsecenter']}
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)

## tests for all possible cases

code to run all test cases and view convergence rates

(can easily modify for loops to run a subset of tests)

In [ ]:
output = run_it(ord_ops=[1],rtype='square',)

In [ ]:
l2,linf,d_ops=run_it(dofloc_ops='node',ord_ops=0,rtype='uniform')

In [ ]:
N = 16
# u = lambda x,y: 5
s = Var(N,dofloc='yside',rtype='square',rname='finecenter',var=u,ords=[0,1])

In [ ]:
s.mesh.view_detailed()

In [ ]:
s.solve_projection(u)

In [ ]:
s.vis_dof_sol(s.operators['mass'].sol_vec,err=False,log=False)
s.vis_dof_sol(s.true_sol_vec,err=False,log=False)

In [ ]:
s.mesh.view_detailed()
s.mesh.patches[1].vis_interface_eval_points()
s.constraints.vis_interface()

In [ ]:
int(-1/2)

let's verify KU = F at corners for p=1, cell center and h=1

In [ ]:
from general_solve.shape_functions import *

In [ ]:
N = 16
u = lambda x,y: x**2/2
f = lambda x,y: 1
# u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
# u = lambda x,y: u0(x) + u0(y)
# f = lambda x,y: -4*np.pi**2*u(x,y)

s = Var(N,dofloc='node',rtype='square',rname='finecenter',var=u,ords=[3,3])

In [ ]:
for p in s.mesh.patches:
	p.vis_interface_eval_points()

In [ ]:
s.constraints.vis_interface()

In [ ]:
s.solve_poisson(f)

In [ ]:
linf_err = np.abs(s.operators['lap'].sol_vec-s.true_sol_vec)

In [ ]:
worst = np.argsort(linf_err)[-20:]

In [ ]:
lhs = (s.constraints.spC.T@s.operators['lap'].spA@s.constraints.spC).todense()
rhs = (s.constraints.spC.T).dot(s.operators['lap'].F)

In [ ]:
corner_sys = corner_pin(lhs,rhs,true_worst_inds,true_vec)

In [ ]:
new_vec,new_lhs,new_rhs,pinned_sys,pinned_vec = corner_sys

In [ ]:
true_worst = np.array([s.true_sol_vec[w] for w in worst])

In [ ]:
new_vec

In [ ]:
np.linalg.norm(true_worst-new_vec)

In [ ]:
[w in s.constraints.true_dofs for w in worst]

In [ ]:
lookup_id = s.mesh.patches[1]._get_lookup_id_from_ind([N/2,N/2])
dof = s.mesh.patches[1].dofs[lookup_id]
print(dof.x,dof.y)

In [ ]:
max(linf_err)

In [ ]:
for w in worst:
	wdof = s.constraints.get_dof(w)
	print(wdof.x,wdof.y,wdof.h==s.h,linf_err[w])

In [ ]:
s.vis_dof_sol(s.operators['lap'].sol_vec,err=False,log=True)

In [ ]:
full_lin_sys = (s.constraints.spC.T @ s.operators['lap'].spA @ s.constraints.spC).todense()
full_rhs = s.constraints.spC.T.dot(s.operators['lap'].F)
corner_integrals = np.array([-1/1680,31/388800,-29/56700,4/42525,-53/100800,
					41/33600,1/1400,59/680400,-409/907200,19/20160,
					-19/322560, 1/230400, 613/268800])
corner_integral_strs =["-1/1680","31/388800","-29/56700","4/42525","-53/100800",
					"41/33600","1/1400","59/680400","-409/907200","19/20160",
					"-19/322560", "1/230400", "613/268800"]
def view_eq(sys,i,j,ufunc=None,p_id=1):
	fig = plt.figure(figsize=(20,20))
	lookup_id = sys.mesh.patches[p_id]._get_lookup_id_from_ind([i,j])
	mydof = sys.mesh.patches[p_id].dofs[lookup_id]
	global_id = sys.constraints._global_dof_id(mydof.ID,p_id)
	true_index = sys.constraints.true_dofs.index(global_id)
	myrow = full_lin_sys[true_index]#.getrow(true_index)
	if abs(full_rhs[true_index]-s.operators['lap'].F[global_id])> 1e-10:
		print('inconsistent f vals')
	fval = full_rhs[true_index]
	print((s.h/2)**2/576+fval)

	if ufunc is not None:
		approx = 0
	for ind in np.nonzero(myrow)[0]:#zip(myrow.indices,myrow.data):
		val = myrow[ind]
		if abs(val) > 1e-10:
			closest_ind = np.argmin(abs(corner_integrals-val))
			if abs(val-corner_integrals[closest_ind])<1e-10:
				ann = corner_integral_strs[closest_ind]
			else:
				ann = round(val,5)

			global_ind = sys.constraints.true_dofs[ind]
			ind_dof = sys.constraints.get_dof(global_ind)
			c = 'C0' if ind_dof.h==sys.h else 'C2'
			plt.plot([ind_dof.x],[ind_dof.y],c+'o',ms=10)
			plt.annotate(ann,(ind_dof.x,ind_dof.y),
				ha='center',va='bottom',fontsize=20)
			if ufunc is not None:
				approx += val*ufunc(ind_dof.x,ind_dof.y)
	if ufunc is not None:
		ftrue = sys.constraints.spC.T.dot(sys.operators['lap'].F)[global_id]
		plt.title('approx: '+str(round(approx,10)),fontsize=20)
			# a=ftrue,b=approx,c=abs(ftrue-approx)),fontsize=30)
	plt.plot([mydof.x],[mydof.y],'C1*',ms=20)
	plt.show()

In [ ]:
# view_eq(s,int(N/2),int(N/2))
view_eq(s,int(N/4)+1,int(N/4)+1,p_id=0)

In [ ]:
# i want to extract the row of the system corresponding to this corner dof
N = 16
u = lambda x,y: x#1/2*(x**4+y**4)
f = lambda x,y: 0#6*x*x+6*y*y

s = Var(N,dofloc='node',rtype='square',rname='finecenter',var=u,ords=[3,3])
s.solve_poisson(f,disp=False)

constraint_coefs = [-19/322560,-19/322560,613/268800,1/230400,-19/322560,1/230400,-19/322560]
fine_coefs = [4/42525,-1/1680,-29/56700,59/680400,-1/1680,-29/56700,1/1400,
 				-409/907200,59/680400,-409/907200,31/388800]
ordered_coefs = np.array(constraint_coefs+fine_coefs)

ordered_coefs = np.array([-1/1680,
 923/85050,
 29/16200,
 -29/28350,
 59/680400,
 41/33600,
 -1787/50400,
 1027/50400,
 19/10080,
 -53/100800,
 19/20160,
 -1649/56700,
 949/50400,
 1/700,
 -409/907200,
 -53/100800,
 13207/1360800,
 619/453600,
 -409/453600,
 31/388800])

full_lin_sys = (s.constraints.spC.T @ s.operators['lap'].spA @ s.constraints.spC).todense()
full_rhs = s.constraints.spC.T.dot(s.operators['lap'].F)

K = s.operators['lap'].spA.todense()

lookup_id = s.mesh.patches[1]._get_lookup_id_from_ind([N/2,N/2])
mydof = s.mesh.patches[1].dofs[lookup_id]
global_id = s.constraints._global_dof_id(mydof.ID,1)
true_index = s.constraints.true_dofs.index(global_id)
myrow = K[global_id]#full_lin_sys[true_index]
mask = abs(myrow)>1e-12


other_dofs = np.arange(K.shape[0])[mask]
u_vals = []
for dof_id in other_dofs:
	ind_dof = s.constraints.get_dof(dof_id)
	u_vals.append(u(ind_dof.x,ind_dof.y))
u_vals = np.array(u_vals)

true_u = s.true_sol_vec
U_comp = true_u[other_dofs]

assert(np.linalg.norm(myrow[mask]-ordered_coefs)<1e-12)
assert(np.linalg.norm(u_vals-U_comp)<1e-12)

LHS = ordered_coefs @ u_vals
RHS = full_rhs[true_index]

print((s.h/2)/24,LHS,RHS)
# assert abs((s.h/2)**2/576+RHS)<1e-12

# print(LHS-RHS)

In [ ]:
11/1440*(s.h/2)**4

In [ ]:
mydof.x,.25-s.h/2

In [ ]:
RHS

In [ ]:
def myrhs(H):
	# return 7*H**3/1440+H**2/384
	return H**4/180+7*H**3/1440+H**2/768

In [ ]:
myrhs(s.h/2)

In [ ]:
U_comp

In [ ]:
u_vals

In [ ]:
ufunc = lambda x,y: x+y#x**2+y**2
# view_eq(s,N/4,N/4,ufunc=ufunc,p_id=0)
# view_eq(s,N,N,ufunc=ufunc)
# view_eq(s,N/2,N/2,ufunc=ufunc)
view_eq(s,N/2,N/2+1,ufunc=ufunc)
# view_eq(s,N/2,N/2+2,ufunc=ufunc)
# view_eq(s,N/2+1,N/2,ufunc=ufunc)
# view_eq(s,N/4+1,N/4+1,ufunc=ufunc,p_id=0)

In [ ]:
for p in s.mesh.patches:
	p.vis()
	p.vis_interface_eval_points()

In [ ]:
s.solve_poisson(f)

In [ ]:
K = s.operators['lap'].spA
F = s.operators['lap'].F
C = s.constraints.spC

In [ ]:
U = s.true_sol_vec

In [ ]:
CTKU = (C.T @ K).dot(U)
CTF = C.T.dot(F)

In [ ]:
xs,ys,vs,fs,ds = [],[],[],[],[]
for true_index,j in enumerate(s.constraints.true_dofs):
	dof = s.constraints.get_dof(j)
	if (s.h < dof.x < 1-s.h) and (s.h<dof.y<1-s.h):
		xs.append(dof.x)
		ys.append(dof.y)
		vs.append(CTKU[true_index])
		fs.append(CTF[true_index])
		ds.append(CTKU[true_index]-CTF[true_index])

In [ ]:
for p in s.mesh.patches:
	p.vis_interface_eval_points()

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(30,8))
cax0 = ax[0].scatter(xs,ys,c=vs,cmap='jet',zorder=1)	
fig.colorbar(cax0)

cax1 = ax[1].scatter(xs,ys,c=fs,cmap='jet',zorder=1)	
fig.colorbar(cax1)

cax2 = ax[2].scatter(xs,ys,c=ds,cmap='jet',zorder=1)	
fig.colorbar(cax2)
plt.show()

In [ ]:
ord_ops = [1,2,3]
for dofloc in doflocs:
	for rname in ['finecenter']:#rnames['square']:
		for ordx in ord_ops:
			for ordy in ord_ops:
				if ordx != ordy:#ordy = ordx
					try:
						s = Var(16,dofloc=dofloc,rtype='square',rname=rname,var=u,ords=[ordx,ordy])
					except:
						print(dofloc,rname,ordx,ordy)

In [ ]:
dofloc = 'cell'
ordx,ordy = 1,1
s = Var(16,dofloc=dofloc,rtype='square',rname=rname,var=u,ords=[ordx,ordy])

In [ ]:
for p in s.mesh.patches:
	p.vis_interface_eval_points()

In [ ]:
s.solve_poisson(f=f)

In [ ]:
.0833333*3

In [ ]:
utest = lambda x,y: x+y
ftest = lambda x,y: 0
for dofloc in ['cell']:#doflocs:
	s = Var(16,dofloc=dofloc,rtype='square',rname='finecenter',var=u,ords=[1,1])
	s.setup_laplace()
	print(s.solve_laplace_truncation(utest,ftest))

## get cell working

In [ ]:
s = Solver(16,2,'cell','square','finecenter',u=u,ords=[1,1])

In [ ]:
s.mesh.view_detailed()
val_list, issue_spots = s.constraints.vis_interface()
print(len(issue_spots))

In [ ]:
s.solve_poisson(f=f)

##### set corners via _locate_corners and extrapolation

In [ ]:
dofloc,N,rname,ord = 'node',16,'finecenter',2

errs = {}
linf = {}
for cornord in [None,0,1,2,3]:#cornord = None
	errs[cornord] = []
	linf[cornord] = []
	for N in [16,32,64]:
		s =	Solver(N,2,dofloc,'square',
				rname=rname,u=u,ords=[ord,ord],
				dirichlet=False,cornord=cornord)
		s.solve_poisson(f=f,disp=False)
		errs[cornord].append(s.lap.err)
		linf[cornord].append(s.lap.Linf_err)
	

In [ ]:
Ns = np.array([16,32,64])

fig = plt.figure(figsize=(20,10))
for index,errors in enumerate([errs,linf]):
	plt.subplot(1,2,index+1)
	for cornord in [None,0,1,2,3]:
		plt.loglog(Ns,errors[cornord],lw=3,label=str(cornord))
	for hord in [1,2,3]:
		plt.loglog(Ns,1/Ns**hord,'-.',lw=3,label='h^{}'.format(hord))
	ttl = 'Linf' if index else 'L2'
	plt.title(ttl,fontsize=20)
	plt.legend(fontsize=20)
plt.show()

^^ these are the convergence rates for p=2 with the corner being handled in four different ways
- None: kept as dof
- 0   : constant extrapolation
- 1   : linear extrapolation
- 2   : quadratic extrapolation
- 3   : cubic extrapolation

we get second order L2 convergence when keeping the corner free and when using constant or linear extrapolation (w constant being a little worse), in these three cases we also get first order Linf convergence

In [ ]:
dofloc,rname,ord = 'node','finecenter',3

errs = {}
linf = {}
for cornord in [None,0,1,2,3]:#cornord = None
	errs[cornord] = []
	linf[cornord] = []
	for N in [16,32,64]:
		s =	Solver(N,2,dofloc,'square',
				rname=rname,u=u,ords=[ord,ord],
				dirichlet=False,cornord=cornord)
		s.solve_poisson(f=f,disp=False)
		errs[cornord].append(s.lap.err)
		linf[cornord].append(s.lap.Linf_err)
	

In [ ]:
Ns = np.array([16,32,64])

fig = plt.figure(figsize=(20,10))
for index,errors in enumerate([errs,linf]):
	plt.subplot(1,2,index+1)
	for cornord in [None,0,1,2,3]:
		plt.loglog(Ns,errors[cornord],lw=3,label=str(cornord))
	for hord in [1,2,3]:
		plt.loglog(Ns,1/Ns**hord,'-.',lw=3,label='h^{}'.format(hord))
	ttl = 'Linf' if index else 'L2'
	plt.title(ttl,fontsize=20)
	plt.legend(fontsize=20)
plt.show()

##### set corners by forcing derivative continuity

In [ ]:
s = Solver(16,2,'node','square','finecenter',u=u,ords=[3,3])

In [ ]:
s.solve_poisson(f=f)

In [ ]:
s.mesh.vis_dof_sol(abs(s.U_true-s.lap.U),true_list=s.constraints.true_dofs)

In [ ]:
for dof_id in s.constraints.Cc:#tocheck[0]:
	if dof_id not in s.constraints.true_dofs:
		print(dof_id)
		dof = s.constraints.get_dof(dof_id)
		if dof_id in s.constraints.corner_mods:
			plt.plot(dof.x,dof.y,'o')
		elif dof_id in s.constraints.all_corner_mods:
			plt.plot(dof.x,dof.y,'s')
		elif dof_id in list(s.constraints.corner_dofs.values()):
			plt.plot(dof.x,dof.y,'*')

		else:
			plt.plot(dof.x,dof.y,'k.')


In [ ]:
# lin extrap for p=2 corners
tmp=run_it(dofloc_ops='node',ord_ops=2,cornord=1)

In [ ]:
# lin extrap for p=2 corners
tmp=run_it(dofloc_ops='node',ord_ops=2,cornord=2)

In [ ]:
# lin extrap for p=3 corners
tmp=run_it(dofloc_ops='node',ord_ops=3)

In [ ]:
# quad extrap for p=3 corners
run_it(dofloc_ops='node',ord_ops=3)

In [ ]:
# cubic extrap for p=3 corners
run_it(dofloc_ops='node',ord_ops=3)

## p=1 for all dofloc options

In [ ]:
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])

def get_convergence(dofloc,ord):
	l2errs = {}
	linferrs = {}
	print('{} centered with order p = {} convergence rates'.format(dofloc,ord))
	print('refinement type\t\tL2\t\tLinf\n'+'-'*50)
	for col,rname in enumerate(rnames['square']):
		print(rname)
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			# if N == 16:
				# s.mesh.view_detailed()
			s.solve_poisson(f=f,disp=False)

			if len(L2)>0:
				print('\t\t\t{}\t\t{}'.format(
					round(L2[-1]/s.lap.err,5),
					round(Linf[-1]/s.lap.Linf_err,5)))
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2errs[rname] = L2
		linferrs[rname] = Linf
	return l2errs,linferrs

def plot_convergence(l2errs,linferrs,dofloc,ord):
	fig = plt.figure(figsize=(30,10))
	for col,rname in enumerate(rnames['square']):
		spot = col+1
		L2 = l2errs[rname]
		Linf = linferrs[rname]
		plt.subplot(1,2,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		plt.title(rname,fontsize=25)
	plt.suptitle('{} centered with order p = {}'.format(dofloc,1),fontsize=40)
	plt.show()

In [ ]:
nodel2,nodelinf = get_convergence('node',1)

In [ ]:
celll2,celllinf = get_convergence('cell',1)

In [ ]:
xsidel2,xsidelinf = get_convergence('xside',1)

In [ ]:
ysidel2,ysidelinf = get_convergence('yside',1)

In [ ]:
dofloc,ord = 'node',1

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	L2,Linf = [],[]
	for N in Nvals:
		s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
		# if N == 16:
			# s.mesh.view_detailed()
		s.solve_poisson(f=f,disp=False)
		L2.append(s.lap.err)
		Linf.append(s.lap.Linf_err)
	l2errs[rname] = L2
	linferrs[rname] = Linf


In [ ]:
fig = plt.figure(figsize=(30,10))
for col,rname in enumerate(rnames['square']):
	spot = col+1
	L2 = l2errs[rname]
	Linf = linferrs[rname]
	plt.subplot(1,2,spot)
	plt.loglog(Nvals,L2,label='L2',lw=5)
	plt.loglog(Nvals,Linf,label='Linf')
	plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
	plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
	plt.legend(fontsize=25)
	plt.title(rname,fontsize=25)
plt.suptitle('{} centered with order p = {}'.format(dofloc,1),fontsize=40)
plt.show()

In [ ]:
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])
dofloc,ord = 'cell',1

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	L2,Linf = [],[]
	for N in Nvals:
		s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
		# if N == 16:
			# s.mesh.view_detailed()
		s.solve_poisson(f=f,disp=False)
		L2.append(s.lap.err)
		Linf.append(s.lap.Linf_err)
	l2errs[rname] = L2
	linferrs[rname] = Linf

In [ ]:
fig = plt.figure(figsize=(30,10))
for col,rname in enumerate(rnames['square']):
	spot = col+1
	L2 = l2errs[rname]
	Linf = linferrs[rname]
	plt.subplot(1,2,spot)
	plt.loglog(Nvals,L2,label='L2',lw=5)
	plt.loglog(Nvals,Linf,label='Linf')
	plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
	plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
	plt.legend(fontsize=25)
	plt.title(rname,fontsize=25)
plt.suptitle('{} centered with order p = {}'.format(dofloc,1),fontsize=40)
plt.show()

In [ ]:
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])
dofloc,ord = 'xside',1

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	L2,Linf = [],[]
	for N in Nvals:
		s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
		# if N == 16:
			# s.mesh.view_detailed()
		s.solve_poisson(f=f,disp=False)
		L2.append(s.lap.err)
		Linf.append(s.lap.Linf_err)
	l2errs[rname] = L2
	linferrs[rname] = Linf

In [ ]:
fig = plt.figure(figsize=(30,10))
for col,rname in enumerate(rnames['square']):
	spot = col+1
	L2 = l2errs[rname]
	Linf = linferrs[rname]
	plt.subplot(1,2,spot)
	plt.loglog(Nvals,L2,label='L2',lw=5)
	plt.loglog(Nvals,Linf,label='Linf')
	plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
	plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
	plt.legend(fontsize=25)
	plt.title(rname,fontsize=25)
plt.suptitle('{} centered with order p = {}'.format(dofloc,1),fontsize=40)
plt.show()

In [ ]:
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])
dofloc,ord = 'yside',1

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	L2,Linf = [],[]
	for N in Nvals:
		s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
		# if N == 16:
			# s.mesh.view_detailed()
		s.solve_poisson(f=f,disp=False)
		L2.append(s.lap.err)
		Linf.append(s.lap.Linf_err)
	l2errs[rname] = L2
	linferrs[rname] = Linf

In [ ]:
fig = plt.figure(figsize=(30,10))
for col,rname in enumerate(rnames['square']):
	spot = col+1
	L2 = l2errs[rname]
	Linf = linferrs[rname]
	plt.subplot(1,2,spot)
	plt.loglog(Nvals,L2,label='L2',lw=5)
	plt.loglog(Nvals,Linf,label='Linf')
	plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
	plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
	plt.legend(fontsize=25)
	plt.title(rname,fontsize=25)
plt.suptitle('{} centered with order p = {}'.format(dofloc,1),fontsize=40)
plt.show()

## test all cases

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])

all_errs = {}

for dofloc in doflocs:
	l2errs = {}
	linferrs = {}
	for col,rname in enumerate(rnames['stripe']):
		l2_tmp = {}
		linf_tmp = {}
		for row,ord in enumerate([1,2,3]):
			L2,Linf = [],[]
			for N in Nvals:
				s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
				s.solve_poisson(f=f,disp=False)
				L2.append(s.lap.err)
				Linf.append(s.lap.Linf_err)
			l2_tmp[ord] = L2
			linf_tmp[ord] = Linf
		l2errs[rname] = l2_tmp
		linferrs[rname] = linf_tmp
	all_errs[dofloc] = (l2errs,linferrs)

In [ ]:
ttls = ['L2 Errors','Linf Errors']
for j in range(2):
	fig = plt.figure(figsize=(40,30))
	for col,rname in enumerate(rnames['stripe']):
		for row,ord in enumerate([1,2,3]):
			spot = 4*row+col+1
			plt.subplot(3,4,spot)
			for dofloc in doflocs:
				errs = all_errs[dofloc][j][rname][ord]
				plt.loglog(Nvals,errs,label=dofloc)
			plt.loglog(Nvals,1/Nvals**(ord+j),label='h^{}'.format(ord+j),lw=5)
			plt.legend(fontsize=25)
			if col == 0:
				plt.ylabel('order {}'.format(ord),fontsize=25)
			if row == 0:
				plt.title(rname,fontsize=25)
	plt.suptitle(ttls[j],fontsize=40)
	plt.show()

# initial tests

## node centered

### not working initially

In [ ]:
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32])
dofloc = 'node'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			# if N == 16:
				# s.mesh.view_detailed()
			s.solve_poisson(f=f,disp=False)
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp


In [ ]:
fig = plt.figure(figsize=(20,30))
for col,rname in enumerate(rnames['square']):
	for row,ord in enumerate([2,3]):
		spot = 2*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(2,2,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		# plt.loglog(Nvals,Linf,label='Linf')
		# plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()

### let's debug

okay! let's reflect on the results for node centered data with square refinement

let's first see where issues are popping up

EEK we have no interface ghosts on our fine mesh for the fine center case

okay that is fixed. we are getting the same sin vs cos issue -- aka we can only handle zero derivatives at refinementttttttt yuck

so the issue ends up being at the corners.

as a note, this issue does not show up in solving L2 projection, we get correct convergence for that test

okay we used extrapolation to remove the corners from the system, but there is still residue at the corners, what if we remove a little more

yikes removing more made things worse, i only removed the two points adjacient to the actual corner

what if instead we just did even higher order extrapolation in the corners, that also made things worse

what if we try linear extrapolation

#### let's try changing how we do the corners

first approach -- extrapolation [4,-6,4,-1] but in 2d

In [ ]:
dofloc,rname,ord = 'node','finecenter',3
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
s = Solver(16,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)

In [ ]:
s = Solver(32,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)

In [ ]:
dofloc,rname,ord = 'node','finecenter',3
u0 = lambda x: np.sin(2*np.pi*x)+np.cos(2*np.pi*x)
u = lambda x,y: u0(x) + u0(y)
f = lambda x,y: -4*np.pi**2*u(x,y)
s = Solver(16,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)

In [ ]:
s.mesh.vis_dof_sol(abs(s.lap.U-s.U_true),true_list=s.constraints.true_dofs)

In [ ]:
dofloc,rname,ord = 'node','coarsecenter',2
u = lambda x,y: np.sin(2*np.pi*x)+np.sin(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
s = Solver(16,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)
e16 = s.lap.err

s = Solver(32,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)
e32 = s.lap.err

plt.loglog([16,32],[e16,e32],label='err')
plt.loglog([16,32],[1/16**3,1/32**3],label='h^3')
plt.legend()
plt.title('convergence rate: {}'.format(e16/e32))
plt.show()

In [ ]:
dofloc,rname,ord = 'node','finecenter',2
u = lambda x,y: np.sin(2*np.pi*x)#+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
s = Solver(16,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)
e16 = s.lap.err

s = Solver(32,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
s.solve_poisson(f=f)
e32 = s.lap.err

plt.loglog([16,32],[e16,e32],label='err')
plt.loglog([16,32],[1/16**3,1/32**3])
plt.legend()
plt.title('convergence rate: {}'.format(e16/e32))
plt.show()

In [ ]:
plt.loglog([16,32],[0.0004675042497614719,5.530804235536717e-05],label='err')
plt.loglog([16,32],[1/16**3,1/32**3])
plt.legend()
plt.show()

In [ ]:
s.mesh.vis_dof_sol(abs(s.lap.U-s.U_true))

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32])
dofloc = 'node'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['square']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([1,2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'square',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)
			# s.solve_projection()
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp

In [ ]:
fig = plt.figure(figsize=(20,30))
for col,rname in enumerate(rnames['square']):
	for row,ord in enumerate([1,2,3]):
		spot = 2*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(3,2,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()

#### fixing this bug!

1.	p=2 horizontal coarse stripe vs vertical coarse stripe, one is working and one is not

In [ ]:
# some plots showing fine center is working

N,dofloc = 16,'node'
pers = [2,4,6,8]
rn_ops = ['vertfinecenter','vertcoarsecenter']
rn_labs = ['fine center', 'coarse center']
for ord in [2,3]:
	linferrs = {rn:[] for rn in rn_ops}
	l2errs = {rn:[] for rn in rn_ops}
	for per in tqdm(pers):
		# print('u(x,y) = cos({} pi x)+cos({} pi y)'.format(per,per))
		for rname in rn_ops:
			# print('\t',rname)
			u = lambda x,y: np.cos(per*np.pi*x)+np.cos(per*np.pi*y)
			f = lambda x,y: -per**2*np.pi**2*u(x,y)
			s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)

			linferrs[rname].append(s.lap.Linf_err)
			l2errs[rname].append(s.lap.err)
			# print('\t\tL2 error: {}\n\t\tLinf error:{}'.format(s.lap.err,s.lap.Linf_err))
			# s.mesh.vis_dof_sol(abs(s.lap.U-s.U_true))

	fig = plt.figure(figsize=(10,3))
	plt.subplot(121)
	plt.title('L2')
	for rn,lab in zip(rn_ops,rn_labs):
		plt.plot(pers,l2errs[rn],label=lab)
	plt.xlabel('oscillations')
	plt.ylabel('error')
	plt.legend()

	plt.subplot(122)
	plt.title('Linf')
	for rn,lab in zip(rn_ops,rn_labs):
		plt.plot(pers,linferrs[rn],label=lab)
	plt.legend()
	plt.xlabel('oscillations')
	plt.ylabel('error')
	plt.suptitle(ord)
	plt.show()

let's look at the constraint matrices for the fine and coarse centers? to some degree they should be the same because fine is always ghost

In [ ]:
# count number of respective coarse and fine dofs in each case

mysolvers = []
coarse_dofs,fine_dofs = [],[]
for rname in ['vertfinecenter','vertcoarsecenter']:
	print(rname)
	print('\tCoarse Count\tFine Count')

	dofloc,ord = 'node',3
	s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
	mysolvers.append(s)

	ftmp,ctmp = [],[]

	fine_dof_count, coarse_dof_count = 0,0
	for dof_id in s.constraints.true_dofs:
		dof = s.constraints.get_dof(dof_id)
		if dof_id < s.constraints.dof_id_shift:
			coarse_dof_count += 1
			ctmp.append(dof)
		else:
			fine_dof_count += 1
			ftmp.append(dof)
	coarse_dofs.append(ctmp)
	fine_dofs.append(ftmp)
	print('\t{}\t{}'.format(coarse_dof_count, fine_dof_count))

In [ ]:
def shift(x):
	if x < .5:
		return x + .5
	else:
		return x - .5

fig,ax = plt.subplots(1,2,figsize=(40,40))
for j,(s,cdofs,fdofs) in enumerate(zip(mysolvers,coarse_dofs,fine_dofs)):
	for c in cdofs:
		x,y = c.x,c.y
		if j==0: 
			ax[0].plot(x,y,'ko')
		else:
			x = shift(x)
			ax[0].plot(x,y,'rx',ms=15)

	for f in fdofs:
		x,y = f.x,f.y
		if j: 
			ax[1].plot(x,y,'ko')
		else:
			x = shift(x)
			ax[1].plot(x,y,'rx',ms=15)
	ax[j].plot([0.25,0.25],[0,1],'grey')
	ax[j].plot([0.75,0.75],[0,1],'grey')
	ax[j].set_aspect('equal')
plt.show()



so there are two columns of fine dofs that are not being set as ghosts when we have coarse center

here's the problem! no ghosts are logged on this patch for some reason

In [ ]:
fstripe,cstripe = mysolvers

fstripe.mesh.patches[1].vis()
cstripe.mesh.patches[1].vis()
cstripe.mesh.patches[1].vis_interface_eval_points()
cstripe.mesh.patches[0].vis_interface_eval_points()

issue is in the refinement class

let's look at stripe_refinement_type 1 and 3 and make sure interface ghosts are being set, should be set on edges in this case

okay far_out was not using interface edges

let's run the checks again

In [ ]:
# count number of respective coarse and fine dofs in each case

mysolvers = []
coarse_dofs,fine_dofs = [],[]
for rname in ['vertfinecenter','vertcoarsecenter']:
	print(rname)
	print('\tCoarse Count\tFine Count')

	dofloc,ord = 'node',3
	s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
	mysolvers.append(s)

	ftmp,ctmp = [],[]

	fine_dof_count, coarse_dof_count = 0,0
	for dof_id in s.constraints.true_dofs:
		dof = s.constraints.get_dof(dof_id)
		if dof_id < s.constraints.dof_id_shift:
			coarse_dof_count += 1
			ctmp.append(dof)
		else:
			fine_dof_count += 1
			ftmp.append(dof)
	coarse_dofs.append(ctmp)
	fine_dofs.append(ftmp)
	print('\t{}\t{}'.format(coarse_dof_count, fine_dof_count))

In [ ]:
fstripe,cstripe = mysolvers

fstripe.mesh.patches[1].vis()
cstripe.mesh.patches[1].vis()
cstripe.mesh.patches[1].vis_interface_eval_points()
cstripe.mesh.patches[0].vis_interface_eval_points()

#### did it work?

### working now

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32,64])
dofloc = 'node'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['stripe']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([1,2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp

In [ ]:
fig = plt.figure(figsize=(40,30))
for col,rname in enumerate(rnames['stripe']):
	for row,ord in enumerate([1,2,3]):
		spot = 4*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(3,4,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()

## cell centered

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32])
dofloc = 'cell'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['stripe']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([1,2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)
			if N == 16:
				s.mesh.view_detailed()
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp

right off the bat, there are some issues with identifying quadrants that have support around the boundaries, not an issue in the centers -- fixed

In [ ]:
fig = plt.figure(figsize=(40,30))
for col,rname in enumerate(rnames['stripe']):
	for row,ord in enumerate([1,2,3]):
		spot = 4*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(3,4,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()

## xside centered

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32])
dofloc = 'xside'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['stripe']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([1,2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)
			if N == 16:
				s.mesh.view_detailed()
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp

In [ ]:
fig = plt.figure(figsize=(40,30))
for col,rname in enumerate(rnames['stripe']):
	for row,ord in enumerate([1,2,3]):
		spot = 4*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(3,4,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()

## yside centered

In [ ]:
u = lambda x,y: np.sin(2*np.pi*x)+np.cos(2*np.pi*y)
f = lambda x,y: -4*np.pi**2*u(x,y)
Nvals = np.array([16,32])
dofloc = 'yside'

l2errs = {}
linferrs = {}
for col,rname in enumerate(rnames['stripe']):
	l2_tmp = {}
	linf_tmp = {}
	for row,ord in enumerate([1,2,3]):
		L2,Linf = [],[]
		for N in Nvals:
			s = Solver(N,2,dofloc,'stripe',rname=rname,u=u,ords=[ord,ord],dirichlet=False)
			s.solve_poisson(f=f,disp=False)
			if N == 16:
				s.mesh.view_detailed()
			L2.append(s.lap.err)
			Linf.append(s.lap.Linf_err)
		l2_tmp[ord] = L2
		linf_tmp[ord] = Linf
	l2errs[rname] = l2_tmp
	linferrs[rname] = linf_tmp

In [ ]:
fig = plt.figure(figsize=(40,30))
for col,rname in enumerate(rnames['stripe']):
	for row,ord in enumerate([1,2,3]):
		spot = 4*row+col+1
		L2 = l2errs[rname][ord]
		Linf = linferrs[rname][ord]
		plt.subplot(3,4,spot)
		plt.loglog(Nvals,L2,label='L2',lw=5)
		plt.loglog(Nvals,Linf,label='Linf')
		plt.loglog(Nvals,1/Nvals**ord,label='h^{}'.format(ord))
		plt.loglog(Nvals,1/Nvals**(ord+1),label='h^{}'.format(ord+1),lw=5)
		plt.legend(fontsize=25)
		if col == 0:
			plt.ylabel('order {}'.format(ord),fontsize=25)
		if row == 0:
			plt.title(rname,fontsize=25)
plt.suptitle(dofloc,fontsize=40)
plt.show()